# Specimen 03 — Coding Agent

Goal: give the model a code-execution tool so it can write a script, run it, see the real output, and fix its own mistakes — the closest thing yet to how the final project's agents will work.

In [1]:
import os
from dotenv import load_dotenv
import anthropic

load_dotenv()
client = anthropic.Anthropic(api_key=os.environ['ANTHROPIC_API_KEY'])
MODEL = 'claude-opus-5'


## 1. Define a run_python tool

Executes a string of Python code (subprocess or a restricted `exec` namespace) and returns stdout/stderr.

In [2]:
import subprocess
import sys
import time

run_python_tool = {
    "name": "run_python",
    "description": "Execute a string of Python 3 code in a sandboxed subprocess and return its stdout and stderr. Use this to compute anything that's easier to solve with code than by reasoning in text.",
    "input_schema": {
        "type": "object",
        "properties": {"code": {"type": "string", "description": "Python source code to execute. Use print() to produce output."}},
        "required": ["code"]
    }
}

FORBIDDEN_PATTERNS = ['os.remove', 'shutil.rmtree', 'socket', 'urllib', 'requests', 'subprocess', 'open(']

def run_python(code, timeout=5):
    for pattern in FORBIDDEN_PATTERNS:
        if pattern in code:
            return {"stdout": "", "stderr": f"Blocked: code contains forbidden pattern '{pattern}'", "returncode": -1}
    try:
        start = time.time()
        result = subprocess.run(
            [sys.executable, "-c", code],
            capture_output=True, text=True, timeout=timeout,
        )
        elapsed = time.time() - start
        return {"stdout": result.stdout, "stderr": result.stderr, "returncode": result.returncode, "elapsed": round(elapsed, 3)}
    except subprocess.TimeoutExpired:
        return {"stdout": "", "stderr": f"Execution timed out after {timeout}s", "returncode": -1}

print(run_python("print(2 + 2)"))

{'stdout': '4\n', 'stderr': '', 'returncode': 0, 'elapsed': 0.031}


## 2. Give it a task easier to solve in code than in text

e.g. 'what's the 50th Fibonacci number' or 'sort this list of 200 random numbers and report the median.'

In [3]:
def coding_agent(user_message, max_tokens=800, max_steps=6, verbose=True):
    tools = [run_python_tool]
    messages = [{"role": "user", "content": user_message}]
    step = 0

    while step < max_steps:
        step += 1
        response = client.messages.create(model=MODEL, max_tokens=max_tokens, tools=tools, messages=messages, thinking={"type": "disabled"})

        if response.stop_reason != 'tool_use':
            final_text = ''.join(b.text for b in response.content if b.type == 'text')
            if verbose:
                print(f'[step {step}] FINAL: {final_text}')
            return final_text

        messages.append({"role": "assistant", "content": response.content})
        tool_results = []
        for block in response.content:
            if block.type == 'tool_use' and block.name == 'run_python':
                code = block.input['code']
                if verbose:
                    print(f'[step {step}] CODE:\n{code}')
                result = run_python(code)
                if verbose:
                    print(f'[step {step}] RESULT: {result}')
                tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": str(result)})
        messages.append({"role": "user", "content": tool_results})

    return None

answer = coding_agent("What is the 50th number in the Fibonacci sequence, where the sequence starts 0, 1, 1, 2, 3...?")
print(f'\nAnswer: {answer}')

[step 1] CODE:

a, b = 0, 1
seq = [a, b]
for _ in range(60):
    a, b = b, a + b
    seq.append(b)

# Indexing: term 1 = 0, term 2 = 1, term 3 = 1, ...
for i in [48, 49, 50, 51]:
    print(f"term {i} (1-indexed) = {seq[i-1]}")
print("F(50) with F(0)=0 convention:", seq[50])

[step 1] RESULT: {'stdout': 'term 48 (1-indexed) = 2971215073\nterm 49 (1-indexed) = 4807526976\nterm 50 (1-indexed) = 7778742049\nterm 51 (1-indexed) = 12586269025\nF(50) with F(0)=0 convention: 12586269025\n', 'stderr': '', 'returncode': 0, 'elapsed': 0.039}


[step 2] FINAL: **7,778,742,049**

Counting the sequence 0, 1, 1, 2, 3, 5, ... with 0 as the 1st number, the 50th number is 7,778,742,049.

A note on conventions: if you instead use the common mathematical indexing where F(0) = 0 and F(1) = 1, then F(50) = 12,586,269,025 — which is the 51st term in the counting above. So which answer you want depends on whether the leading 0 is called the 1st term or the 0th.

Answer: **7,778,742,049**

Counting the sequence 0, 1, 1, 2, 3, 5, ... with 0 as the 1st number, the 50th number is 7,778,742,049.

A note on conventions: if you instead use the common mathematical indexing where F(0) = 0 and F(1) = 1, then F(50) = 12,586,269,025 — which is the 51st term in the counting above. So which answer you want depends on whether the leading 0 is called the 1st term or the 0th.


## 3. Let it see its own errors

Pick a task where its first attempt is likely to throw (an edge case). Confirm it reads the traceback and fixes the code rather than giving up.

In [4]:
error_prone_task = (
    "Write Python code that computes the median of this list of numbers: "
    "[4, 8, 15, 16, 23, 42]. First try computing it with a deliberately broken approach "
    "that assumes an odd-length list (so it will throw an index error on this even-length list), "
    "observe the error, then fix your code and give me the correct median."
)

answer = coding_agent(error_prone_task)
print(f'\nAnswer: {answer}')

[step 1] CODE:

data = [4, 8, 15, 16, 23, 42]

def median_broken(nums):
    s = sorted(nums)
    n = len(s)
    # Deliberately broken: assumes odd length, so the "middle" index is n//2
    # and it also (wrongly) tries to index n to grab a "second middle"
    mid = n // 2
    return (s[mid] + s[n]) / 2   # s[n] is out of range -> IndexError

print(median_broken(data))

[step 1] RESULT: {'stdout': '', 'stderr': 'Traceback (most recent call last):\n  File \x1b"<string>"\x1b, line \x1b12\x1b, in \x1b<module>\x1b\n    print(\x1bmedian_broken\x1b\x1b(data)\x1b)\n          \x1b~~~~~~~~~~~~~\x1b\x1b^^^^^^\x1b\n  File \x1b"<string>"\x1b, line \x1b10\x1b, in \x1bmedian_broken\x1b\n    return (s[mid] + \x1bs\x1b\x1b[n]\x1b) / 2   # s[n] is out of range -> IndexError\n                     \x1b~\x1b\x1b^^^\x1b\n\x1bIndexError\x1b: \x1blist index out of range\x1b\n', 'returncode': 1, 'elapsed': 0.061}


[step 2] CODE:

import statistics

data = [4, 8, 15, 16, 23, 42]

def median(nums):
    if not nums:
        raise ValueError("median of empty list is undefined")
    s = sorted(nums)
    n = len(s)
    mid = n // 2
    if n % 2 == 1:
        return s[mid]
    return (s[mid - 1] + s[mid]) / 2

print("sorted:          ", sorted(data))
print("two middles:     ", sorted(data)[2], sorted(data)[3])
print("median (mine):   ", median(data))
print("median (stdlib): ", statistics.median(data))

# sanity check on an odd-length list too
print("odd-length check:", median([4, 8, 15, 16, 23]), statistics.median([4, 8, 15, 16, 23]))

[step 2] RESULT: {'stdout': 'sorted:           [4, 8, 15, 16, 23, 42]\ntwo middles:      15 16\nmedian (mine):    15.5\nmedian (stdlib):  15.5\nodd-length check: 15 15\n', 'stderr': '', 'returncode': 0, 'elapsed': 0.064}


[step 3] FINAL: ## What happened

**Broken attempt:** It computed `mid = n // 2` and then tried `(s[mid] + s[n]) / 2`. For a 6-element list, `n` is 6 but valid indices only go up to 5, so it raised `IndexError: list index out of range`.

**The fix:** Check the parity of the length. For an odd length, return the single middle element `s[n//2]`. For an even length, average the two middle elements, `s[n//2 - 1]` and `s[n//2]` — note the `- 1`, which is what the broken version got wrong.

```python
def median(nums):
    if not nums:
        raise ValueError("median of empty list is undefined")
    s = sorted(nums)
    n = len(s)
    mid = n // 2
    if n % 2 == 1:
        return s[mid]
    return (s[mid - 1] + s[mid]) / 2

print(median([4, 8, 15, 16, 23, 42]))  # 15.5
```

**The median is 15.5.** The list is already sorted and has 6 elements, so the two middle values are 15 and 16, and their average is 15.5. This matches `statistics.median`, and the function also gives the right answer on 

## 4. Sandbox it for real

Set an execution timeout and block dangerous operations (file deletion, network calls) before running anything the model writes.

In [5]:
print('Timeout test:')
print(run_python("import time; time.sleep(10); print('done')", timeout=2))

print('\nBlocked operation test:')
print(run_python("import os; os.remove('some_file.txt')"))

print('\nNormal execution still works:')
print(run_python("print(sum(range(100)))"))

Timeout test:


{'stdout': '', 'stderr': 'Execution timed out after 2s', 'returncode': -1}

Blocked operation test:
{'stdout': '', 'stderr': "Blocked: code contains forbidden pattern 'os.remove'", 'returncode': -1}

Normal execution still works:
{'stdout': '4950\n', 'stderr': '', 'returncode': 0, 'elapsed': 0.146}


## 5. Compare against Phase 1-2 knowledge

Give it a small ML task (e.g. 'fit a linear regression on this data and report the R²'). Does it reach for the right library on its own?

In [6]:
ml_task = (
    "I have x = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10] and y = [2.1, 3.9, 6.2, 7.8, 10.1, 12.3, 13.9, 16.2, 17.8, 20.1]. "
    "Write and run Python code to fit a linear regression and report the slope, intercept, and R-squared."
)

answer = coding_agent(ml_task)
print(f'\nAnswer: {answer}')

[step 1] CODE:

import numpy as np

x = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], dtype=float)
y = np.array([2.1, 3.9, 6.2, 7.8, 10.1, 12.3, 13.9, 16.2, 17.8, 20.1])

n = len(x)
slope, intercept = np.polyfit(x, y, 1)

y_pred = slope * x + intercept
ss_res = np.sum((y - y_pred) ** 2)
ss_tot = np.sum((y - y.mean()) ** 2)
r2 = 1 - ss_res / ss_tot

# standard errors
s_err = np.sqrt(ss_res / (n - 2))
se_slope = s_err / np.sqrt(np.sum((x - x.mean()) ** 2))
se_intercept = s_err * np.sqrt(1/n + x.mean()**2 / np.sum((x - x.mean())**2))

print(f"slope      = {slope:.6f}  (SE {se_slope:.6f})")
print(f"intercept  = {intercept:.6f}  (SE {se_intercept:.6f})")
print(f"R-squared  = {r2:.6f}")
print(f"r (corr)   = {np.corrcoef(x, y)[0,1]:.6f}")
print(f"residual SD= {s_err:.6f}")
print("\nresiduals:", np.round(y - y_pred, 4))



[step 1] RESULT: {'stdout': 'slope      = 1.998788  (SE 0.020739)\nintercept  = 0.046667  (SE 0.128684)\nR-squared  = 0.999139\nr (corr)   = 0.999570\nresidual SD= 0.188374\n\nresiduals: [ 0.0545 -0.1442  0.157  -0.2418  0.0594  0.2606 -0.1382  0.163  -0.2358\n  0.0655]\n', 'stderr': '', 'returncode': 0, 'elapsed': 0.278}


[step 2] FINAL: ## Results

| Quantity | Value |
|---|---|
| **Slope** | **1.9988** (± 0.0207) |
| **Intercept** | **0.0467** (± 0.1287) |
| **R²** | **0.9991** |

**Fitted line:** y = 1.9988x + 0.0467

**Interpretation:**
- The relationship is essentially perfectly linear — R² = 0.9991 means 99.91% of the variance in y is explained by x.
- The slope of ~2.0 says y increases by about 2 units for every 1-unit increase in x.
- The intercept of 0.047 is small relative to its standard error (0.129), so it's not statistically distinguishable from zero. Your data is consistent with the simpler relationship **y ≈ 2x**.
- Residuals are small (SD ≈ 0.19) and alternate in sign in a regular pattern, which is just the rounding structure of your y-values rather than any real model misfit.

Answer: ## Results

| Quantity | Value |
|---|---|
| **Slope** | **1.9988** (± 0.0207) |
| **Intercept** | **0.0467** (± 0.1287) |
| **R²** | **0.9991** |

**Fitted line:** y = 1.9988x + 0.0467

**Interpretation:

## 6. Write a short retrospective

3-5 sentences: what did giving it code execution add that specimen 02's individual tools didn't?

In [7]:
retrospective = """
Specimen 02's ReAct loop could only ever call the specific tools I pre-wrote (a calculator, a price lookup) —
each one solves exactly one narrow, anticipated problem. This coding agent has a single tool, run_python, but
that one tool can solve an open-ended range of problems because the model gets to write the logic itself
instead of being limited to calling functions I already built. The real difference showed up in step 3: when
the deliberately broken code threw an IndexError, the agent read the actual traceback and rewrote its own
approach, something no fixed tool set could do since none of my Phase 4 tools return code-shaped errors for
the model to reason about. The tradeoff is that this specimen needs a much more serious sandbox than specimen
02 did, since arbitrary code is a far bigger attack surface than a fixed handful of narrow functions.
"""

print(retrospective.strip())

Specimen 02's ReAct loop could only ever call the specific tools I pre-wrote (a calculator, a price lookup) —
each one solves exactly one narrow, anticipated problem. This coding agent has a single tool, run_python, but
that one tool can solve an open-ended range of problems because the model gets to write the logic itself
instead of being limited to calling functions I already built. The real difference showed up in step 3: when
the deliberately broken code threw an IndexError, the agent read the actual traceback and rewrote its own
approach, something no fixed tool set could do since none of my Phase 4 tools return code-shaped errors for
the model to reason about. The tradeoff is that this specimen needs a much more serious sandbox than specimen
02 did, since arbitrary code is a far bigger attack surface than a fixed handful of narrow functions.
